# Harringtonine CropArray Example

---

## Notebook summary 


- Load a microscope image of video
- Tracking spots on the image and generate a pandas dataframe with the spots locations
- Creating a croparray with the image and dataframe
- Signal quantification and plotting
- Visualization of croparray with Napari

----

## Importing libraries


----

In [1]:
# To manipulate arrays
import numpy as np 
from skimage.io import imread
import matplotlib.pyplot as plt 
from matplotlib.path import Path
import pylab as pyl
import seaborn as sns; sns.set()  
import pathlib # for working with windows paths
import sys
import cv2
import os
import glob
import trackpy as tp
import pandas as pd
#!pip install shapely
from shapely.geometry import Polygon 
from shapely.geometry import Point
current_dir = pathlib.Path().absolute()
croparray_dir = current_dir.parents[0].joinpath('croparray')
sys.path.append(str(croparray_dir))
import crop_array_tools as ca
import os
import xarray as xr
# %matplotlib inline 
plt.style.use('dark_background')

# Napari 
%gui qt5 
import napari
from napari.utils import nbscreenshot

# Magicgui
from magicgui import magicgui
import datetime
import pathlib
import tifffile as tifffile

## Functions 

___

In [3]:
def TrackWithMasks(img_max, particle_diameter ,min_m, mask_include, mask_exclude):
    f = tp.batch(img_max, diameter=particle_diameter,minmass=min_m)
    f_list = []
    for i in np.arange(len(f['frame'].unique())):
        f0 = f[f['frame']==i]
        f1 = f0.copy()

        # If no masks, include everything and exclude nothing
        if len(mask_include) == 0 :
            mask_include1 = [[0,0],[0,10000000],[10000000,10000000],[10000000,0]]
        else:
            mask_include1 = mask_include
        if len(mask_exclude) == 0:
            mask_exclude1 = [[0,0],[0,-10],[-10,-10]]
        else:
            mask_exclude1 = mask_exclude

        mask_in = Polygon(mask_include1)  # This is a polygon that defines the mask
        mypts = np.transpose([f1.y,f1.x]) # These are the points detected by trackpy, notice the x/y inversion for napari
        f1['Include']=[mask_in.contains(Point(mypts[i])) for i in np.arange(len(mypts))] # Check if pts are on/in polygon mask  

        # # Label points in nucleus if polygon mask exists
        # if polygon != None:
        mask_out = Polygon(mask_exclude1)  # This is a polygon that defines the mask
        f1['Exclude']=[mask_out.contains(Point(mypts[i])) for i in np.arange(len(mypts))] # Check if pts are on/in polygon mask  
        
        f_list.append(f1[(f1['Include']==True) & (f1['Exclude']==False)])
    f_all = pd.concat(f_list)
    return f_all

In [4]:
def TrackWithMasks1(img_max, particle_diameter, min_m, mask_include, mask_exclude, correction_factor, threshold_multiplier):
    f_list = []
    for i in np.arange(img_max.shape[0]):  # Loop through each timepoint
        threshold = threshold_multiplier * correction_factor[i, :, 1].mean()  # Channel 1 is translation

        # Detect particles using Trackpy with the dynamic threshold
        f0 = tp.locate(img_max[i], diameter=particle_diameter, minmass=min_m, threshold=threshold)

        if f0.empty:
            print(f"No particles found in frame {i}.")
            continue

        f1 = f0.copy()

        # Add 'frame' column manually
        f1['frame'] = i

        # If no masks, include everything and exclude nothing
        if len(mask_include) == 0:
            mask_include1 = [[0, 0], [0, 10000000], [10000000, 10000000], [10000000, 0]]
        else:
            mask_include1 = mask_include

        if len(mask_exclude) == 0:
            mask_exclude1 = [[0, 0], [0, -10], [-10, -10]]
        else:
            mask_exclude1 = mask_exclude

        # Create inclusion and exclusion masks
        mask_in = Polygon(mask_include1)
        mask_out = Polygon(mask_exclude1)
        mypts = np.transpose([f1.y, f1.x])  # Trackpy uses x/y, napari uses y/x
        
        # Filter by masks
        f1['Include'] = [mask_in.contains(Point(pt)) for pt in mypts]
        f1['Exclude'] = [mask_out.contains(Point(pt)) for pt in mypts]

        # Append to the list, filtering spots based on masks
        f_list.append(f1[(f1['Include'] == True) & (f1['Exclude'] == False)])

    # Concatenate all frames into a single DataFrame
    f_all = pd.concat(f_list)

    return f_all


In [5]:
def calculate_correction_factors(movie, N):
    """
    Calculate correction factors across all frames to match the total intensity
    of each z-plane to that of the specified frame N, using NumPy array operations.
   
    Parameters:
    - movie: numpy array with dimensions (t, z, x, y, ch),
             where t = timepoints, z = z-planes, x and y = spatial dimensions, ch = channels.
    - N: int, the frame index used as the reference for intensity matching.
   
    Returns:
    - correction_factors: numpy array with correction factors.
    """
    # Calculate the mean intensity for each z-plane and channel at each timepoint
    mean_intensities = movie.mean(axis=(2, 3))
   
    # Extract the mean intensity for the reference frame N for each z-plane and channel
    reference_intensities = mean_intensities[N]
   
    # Calculate the correction factors for each timepoint, z-plane, and channel
    correction_factors = np.ones_like(mean_intensities)  # Initialize to ones

    # Correct channel 0 using the reference intensity at N
    correction_factors[:, :, 0] = reference_intensities[:, 0] / mean_intensities[:, :, 0]

    # If there are more than 1 channel, correct channel 1 using the max mean value in the first 5 time points
    if movie.shape[-1] > 1:
        # Extract the mean intensity for the first 5 frames for each z-plane and channel 1
        mean_intensities_first_5 = mean_intensities[:5]
        # Find the maximum average value within the first 5 time points for each z-plane and channel 1
        max_mean_intensity_first_5_ch1 = mean_intensities_first_5[:, :, 1].max(axis=0)
        correction_factors[:, :, 1] = max_mean_intensity_first_5_ch1 / mean_intensities[:, :, 1]

    # If there are more than 2 channels, correct channel 2 using the reference intensity at N
    if movie.shape[-1] > 2:
        correction_factors[:, :, 2] = reference_intensities[:, 2] / mean_intensities[:, :, 2]
   
    # Avoid division by zero by setting any infinite correction factors to 1
    correction_factors[np.isinf(correction_factors)] = 1
   
    return correction_factors

In [6]:
import scipy.ndimage as ndi

def smooth_based_on_correction(movie, correction_factors):
    """
    Apply a Gaussian blur to each frame of the movie based on the correction factor.
   
    Parameters:
    - movie: numpy array with dimensions (t, z, x, y, ch),
             where t = timepoints, z = z-planes, x and y = spatial dimensions, ch = channels.
    - correction_factors: numpy array with dimensions (t, z, ch),
                          where t = timepoints, z = z-planes, ch = channels.
   
    Returns:
    - smoothed_movie: numpy array with smoothed frames.
    """
    # Initialize an array to hold the smoothed movie
    smoothed_movie = np.empty_like(movie)

    # Loop over each frame in the movie
    for t in range(movie.shape[0]):
        for z in range(movie.shape[1]):
            for ch in range(movie.shape[4]):
                # Calculate the sigma for the Gaussian blur based on the correction factor
                sigma = correction_factors[t, z, ch] / 3 # Divide by 3 for a smoother blur

                # Apply the Gaussian blur to the frame
                smoothed_movie[t, z, :, :, ch] = ndi.gaussian_filter(movie[t, z, :, :, ch], sigma=sigma)

    return smoothed_movie

## Calculate Correction Factor on Raw Movie

In [1308]:
cellname = 'Cell014'

In [1309]:
dir_raw = r"Z:\koch\SARS-CoV-2 Project\Imaging\Terminator Imaging\Rapid\Nsp1_Emi_NoStart\20260112\65msExp_20T_0I\Dish01"

In [1310]:
raw_img_filename = f"{cellname}.tif"
raw_img = imread(os.path.join(dir_raw,raw_img_filename))
raw_img.shape

(20, 13, 512, 512, 3)

In [1311]:
img_raw1 = raw_img.transpose([0,1,2,3,4])
print("Correction factor calculation format shape [f, z, y, x, ch] = ", img_raw1.shape)

Correction factor calculation format shape [f, z, y, x, ch] =  (20, 13, 512, 512, 3)


In [1312]:
correction_factors = calculate_correction_factors(img_raw1, 0)

## Loading in corrected transformed movie


----

In [1313]:
dir = r"Z:\koch\SARS-CoV-2 Project\Imaging\Terminator Imaging\Rapid\Nsp1_Emi_NoStart\20260112\65msExp_20T_0I\Dish01"
my_output_dir = dir

In [1314]:
#Video directory
img_4D_filename = f"{cellname}.tif"
img_4D =  imread(os.path.join(dir,img_4D_filename))
img_4D.shape

(20, 13, 512, 512, 3)

### Smooth movie based on correction factors

In [1315]:
img_4D = img_4D.transpose([0,1,2,3,4])
img_4D.shape
img_4D1 = smooth_based_on_correction(img_4D, correction_factors)
img_4D = img_4D1.transpose([0,1,2,3,4])
img_4D.shape

(20, 13, 512, 512, 3)

In [1316]:
# Converting the video to Croparray format
img_croparray = np.expand_dims(img_4D,axis=0) # 
img_croparray.shape # dimensions MUST be (fov, f , z, y, x, ch)
img_croparray.shape

#if you need to reorder dimensions
img_croparray1 = np.expand_dims(img_4D,axis=0) 
img_croparray = img_croparray1.transpose([0,1,2,3,4,5]) # dimensions MUST be (fov, f , z, y, x, ch)

print("croparray format shape [fov, f , z, y, x, ch] = ", img_croparray.shape)

croparray format shape [fov, f , z, y, x, ch] =  (1, 20, 13, 512, 512, 3)


### Translation Tracking

In [938]:
viewer1 = napari.view_image(img_croparray[0,:,:,:,:,1])

In [ ]:
best_zs = [0,6]
img_max = np.max(img_croparray[0,:,best_zs[0]:best_zs[1],:,:,1],axis=1)
img_max.shape

In [ ]:
viewer2 = napari.view_image(np.max(img_max,axis=0))  # Max of max, should be a single frame

In [ ]:
# First adjust the particle diameter and min mass to detect particles best in the first frame
# Then adjust the threshold multiplier to reduce the number of false positives in the later frames
particle_diameter = 5
min_mass  = 4000
threshold_multiplier = 175
viewer3 = napari.view_image(img_max)
f_all = TrackWithMasks1(img_max,particle_diameter,min_mass,viewer2.layers['Shapes [1]'].data[0][:,-2:],viewer2.layers['Shapes'].data[0][:,-2:],correction_factors,threshold_multiplier)
data = f_all[['frame','y','x']].values
layer_name = 'Spots all frames'
viewer3.add_points(data, size = 12, face_color='transparent', edge_color = 'yellow', symbol='disc', name = layer_name)

In [ ]:
f_all.to_csv(os.path.join(my_output_dir,img_4D_filename[:-4]+"_Translation"+'.csv'))

In [ ]:
#only if you didn't track:
spots = f_all.copy() # Nice to copy; seems it can cause to overwrite otherwise
spots['id']=spots.index
spots.rename(columns={'x': 'xc','y': 'yc', 'frame': 'f','signal':'signal_tp'}, 
inplace=True, errors='raise')
spots['fov']=0
spots.rename(columns={'particle':'id'})
spots = spots[['fov','id','f','yc','xc','signal_tp','Include','Exclude']] # keeping signal out of curiousity... want to compare to disk-donut measurements  
spots.head()

In [ ]:
my_ca = ca.create_crop_array(img_croparray,spots)#,xy_pad=5, dxy=130, dz=500, dt=1, units=['nm','min'], name = os.path.join(dir,img_4D_filename))
my_ca

In [ ]:
my_ca.to_netcdf(os.path.join(my_output_dir,img_4D_filename[:-4]+"_Translation"+'.nc') )

### mRNA Tracking

In [1317]:
viewer1 = napari.view_image(img_croparray[0,:,:,:,:,0])

In [1318]:
best_zs = [4,10]
img_max = np.max(img_croparray[0,:,best_zs[0]:best_zs[1],:,:,0],axis=1)
img_max.shape

(20, 512, 512)

Now you can create a mask to exclude regions: Shapes will be excluded and Shapes [1] will be included in tracks

In [1319]:
viewer2 = napari.view_image(np.max(img_max,axis=0))  # Max of max, should be a single frame

In [1322]:
# Use this if you don't have a mask:
# f_all = TrackWithMasks(img_max,7,1000,[],[])
particle_diameter = 5
min_mass = 2200
viewer3 = napari.view_image(img_max)
f_all = TrackWithMasks(img_max,particle_diameter,min_mass,viewer2.layers['Shapes [1]'].data[0][:,-2:],viewer2.layers['Shapes'].data[0][:,-2:])
data = f_all[['frame','y','x']].values
layer_name = 'Spots all frames'
viewer3.add_points(data, size = 12, face_color='transparent', edge_color = 'yellow', symbol='disc', name = layer_name)

Frame 19: 196 features


<Points layer 'Spots all frames' at 0x20d1506e220>

In [1323]:
f_all.to_csv(os.path.join(my_output_dir,img_4D_filename[:-4]+"_mRNA"+'.csv'))

Convert f to crop_array format

In [1324]:
#only if you didn't track:
spots = f_all.copy() # Nice to copy; seems it can cause to overwrite otherwise
spots['id']=spots.index
spots.rename(columns={'x': 'xc','y': 'yc', 'frame': 'f','signal':'signal_tp'}, 
inplace=True, errors='raise')
spots['fov']=0
spots.rename(columns={'particle':'id'})
spots = spots[['fov','id','f','yc','xc','signal_tp','Include','Exclude']] # keeping signal out of curiousity... want to compare to disk-donut measurements  
spots.head()

,fov,id,f,yc,xc,signal_tp,Include,Exclude
3,0,3,0,146.773271,64.826304,740.996188,True,False
4,0,4,0,153.979461,107.014811,285.029683,True,False
5,0,5,0,157.944776,122.775906,530.020567,True,False
6,0,6,0,175.281912,82.041127,763.217469,True,False
7,0,7,0,177.324152,107.875887,368.397449,True,False


In [1325]:
my_ca = ca.create_crop_array(img_croparray,spots)#,xy_pad=5, dxy=130, dz=500, dt=1, units=['nm','min'], name = os.path.join(dir,img_4D_filename))
my_ca

Original video dimensions:  (1, 20, 13, 512, 512, 3)
Padded video dimensions:  (1, 20, 13, 524, 524, 3)
Max # of spots per frame:  226
Shape of numpy array to hold all crop intensity data:  (1, 226, 20, 13, 11, 11, 3)
Shape of xc and yc numpy arrays:  (1, 226, 20, 3)
Shape of extra my_layers numpy array:  (4, 1, 226, 20)


<xarray.Dataset>
Dimensions:    (fov: 1, n: 226, t: 20, z: 13, y: 11, x: 11, ch: 3)
Coordinates:
  * fov        (fov) int32 0
  * n          (n) int16 0 1 2 3 4 5 6 7 8 ... 218 219 220 221 222 223 224 225
  * t          (t) int32 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19
  * z          (z) int32 0 1 2 3 4 5 6 7 8 9 10 11 12
  * y          (y) int32 -5 -4 -3 -2 -1 0 1 2 3 4 5
  * x          (x) int32 -5 -4 -3 -2 -1 0 1 2 3 4 5
  * ch         (ch) int32 0 1 2
Data variables:
    int        (fov, n, t, z, y, x, ch) int32 7503 6975 5291 6823 ... 0 0 0 0
    xc         (fov, n, t, ch) int32 71 71 71 73 73 73 69 69 ... 0 0 0 0 0 0 0 0
    yc         (fov, n, t, ch) int32 153 153 153 155 155 155 155 ... 0 0 0 0 0 0
    dx         int32 1
    dy         int32 1
    dz         int32 1
    dt         int32 1
    xy_pad     int32 5
    id         (fov, n, t) float64 3.0 230.0 450.0 669.0 ... 0.0 0.0 0.0 0.0
    signal_tp  (fov, n, t) float64 741.0 586.0 800.0 619.0 ... 0.0 0.0 0.0 0.0
    Include    (fov, n, t) float64 1.0 1.0 1.0 1.0 1.0 ... 0.0 0.0 0.0 0.0 0.0
    Exclude    (fov, n, t) float64 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Attributes:
    name:     video_filename
    date:     video_date

In [ ]:
my_ca.to_netcdf(os.path.join(my_output_dir,img_4D_filename[:-4]+"_mRNA"+'.nc') )

: 